# Lab5.3 — CPU vs PL: comparația rezultatului


Comparăm resize-ul OpenCV executat pe ARM cu resize-ul bilinear executat de acceleratorul FPGA.


Pregătim Overlay-ul hardware Resize.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

OVERLAY_DIR = Path("overlay")
OVERLAY_DIR.mkdir(exist_ok=True)
LOCAL_BIT = OVERLAY_DIR / "resizer.bit"
LOCAL_HWH = OVERLAY_DIR / "resizer.hwh"

def find_resizer_overlay():
    if LOCAL_BIT.exists() and LOCAL_HWH.exists():
        return LOCAL_BIT
    try:
        import pynq_helloworld
        pkg = Path(pynq_helloworld.__file__).resolve().parent / "notebooks" / "edge"
        pkg_bit = pkg / "resizer.bit"
        pkg_hwh = pkg / "resizer.hwh"
        if pkg_bit.exists() and pkg_hwh.exists():
            return pkg_bit
    except ImportError:
        pass
    urls = {
        LOCAL_BIT: "https://www.xilinx.com/bin/public/openDownload?filename=pynqhelloworld.resizer.pynqz2.bit",
        LOCAL_HWH: "https://www.xilinx.com/bin/public/openDownload?filename=pynqhelloworld.resizer.pynqz2.hwh",
    }
    for path, url in urls.items():
        if not path.exists():
            print(f"Downloading {path.name} ...")
            urlretrieve(url, path)
    return LOCAL_BIT

RESIZER_BIT = find_resizer_overlay()
print("Overlay:", RESIZER_BIT)

Localizăm imaginea comună folosită în Lab4 și Lab5.


In [ ]:
from pathlib import Path

IMAGE_PATH = Path("../Lab4/images/fruits.jpg")
if not IMAGE_PATH.exists():
    raise FileNotFoundError("Nu a fost găsită imaginea ../Lab4/images/fruits.jpg")
print("Image:", IMAGE_PATH)

Încărcăm imaginea, Overlay-ul și pregătim dimensiunile de lucru.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
old_h, old_w, _ = image_bgr.shape
new_w, new_h = 320, 240

overlay = Overlay(str(RESIZER_BIT))
dma = overlay.axi_dma_0
resizer = overlay.resize_accel_0

Calculăm referința software folosind interpolare biliniară OpenCV.


In [ ]:
cpu_bgr = cv2.resize(image_bgr, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
print("CPU result:", cpu_bgr.shape)

Executăm aceeași redimensionare în PL cu DMA și acceleratorul hardware.


In [ ]:
in_buffer = allocate(shape=image_bgr.shape, dtype=np.uint8)
out_buffer = allocate(shape=(new_h, new_w, 3), dtype=np.uint8)
np.copyto(in_buffer, image_bgr)

resizer.register_map.src_rows = old_h
resizer.register_map.src_cols = old_w
resizer.register_map.dst_rows = new_h
resizer.register_map.dst_cols = new_w

dma.sendchannel.transfer(in_buffer)
dma.recvchannel.transfer(out_buffer)
resizer.write(0x00, 0x81)
dma.sendchannel.wait()
dma.recvchannel.wait()

pl_bgr = np.array(out_buffer)

Afișăm rezultatele CPU și PL la aceeași dimensiune pentru a compara vizual calitatea.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(cv2.cvtColor(cpu_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("OpenCV / ARM")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(pl_bgr, cv2.COLOR_BGR2RGB))
axes[1].set_title("Resize accelerator / PL")
axes[1].axis("off")
plt.tight_layout()
plt.show()

Calculăm diferența pixel-cu-pixel dintre rezultatul software și rezultatul hardware.


In [ ]:
diff = cv2.absdiff(cpu_bgr, pl_bgr)
print("Mean absolute difference:", float(diff.mean()))
print("Maximum difference      :", int(diff.max()))
print("Exactly equal values    :", f"{100*np.mean(cpu_bgr == pl_bgr):.2f}%")

Rezultatele nu trebuie să fie neapărat identice bit-cu-bit. Implementarea OpenCV și acceleratorul HLS pot utiliza rotunjiri și aritmetică internă diferite, chiar dacă ambele folosesc interpolare biliniară.


Eliberăm bufferele DMA.


In [ ]:
del in_buffer, out_buffer


## Întrebări

1. De ce pot exista diferențe numerice între CPU și PL?
2. Este o diferență de 1–2 niveluri pe canal neapărat relevantă vizual?
3. Ce metrică ați folosi pentru o comparație mai riguroasă a calității?
